**NOTICE:**  
The U.S. Army Corps of Engineers, Risk Management Center (USACE-RMC) makes no guarantees about the results, or appropriateness of outputs, obtained from Numerics.

**License:** BSD-style USACE-RMC License (see LICENSE file)

# 00. Getting Started with Numerics in Python
Welcome! This notebook with guide you throught setting up and using the USACE-RMC Numerics library in Python via PythonNet.

## What is Numerics?
Numerics is a comprehensive .NET library for numerical computing and statistical analysis, developed by the U.S. Army Corps of Engineers Risk Management Center. It includes:

- **42+ probability distributions** with advanced fitting methods
- **Bayesian MCMC samplers** (RWMH, DEMCz, HMC, and more)
- **Machine learning** algorithms (Random Forest, K-Means, GMM)
- **Optimization** routines (global and local)
- **Statistical tests** and bootstrap methods
- **Time series** analysis tools

## Obtaining Numerics from USACE-RMC GitHub

There are two recommended ways to get the Numerics library:

### Option 1: Download the Repository Directly

1. Visit the [USACE-RMC/Numerics GitHub repository](https://github.com/USACE-RMC/Numerics)
2. Click the green **"Code"** button and select **"Download ZIP"**
3. Extract the ZIP file to a location on your computer
4. Build the solution (requires Visual Studio or .NET SDK):
   ```bash
   cd Numerics
   dotnet build Numerics.sln --configuration Release
   ```
5. The compiled `Numerics.dll` will be in `Numerics/bin/Release/net8.0/` (or similar)

### Option 2: Clone with Git (Recommended for Auto-Updates)

This approach allows you to easily pull the latest changes without re-downloading everything.

1. Install Git (if not already installed): [git-scm.com](https://git-scm.com/)
2. Clone the repository:
   ```bash
   git clone https://github.com/USACE-RMC/Numerics.git
   cd Numerics
   ```
3. Build the DLL:
   ```bash
   dotnet build Numerics.sln --configuration Release
   ```
4. Update to latest version anytime:
   ```bash
   cd Numerics  # Navigate to the cloned directory
   git pull origin main  # Pull latest changes
   dotnet build Numerics.sln --configuration Release  # Rebuild
   ```

**Pro Tip:** Create a shell script or batch file to automate the update and rebuild:

**Linux/Mac** (`update_numerics.sh`):
```bash
#!/bin/bash
cd /path/to/Numerics
git pull origin main
dotnet build Numerics.sln --configuration Release
echo "Numerics updated and rebuilt successfully!"
```

**Windows** (`update_numerics.bat`):
```batch
@echo off
cd C:\\path\\to\\Numerics
git pull origin main
dotnet build Numerics.sln --configuration Release
echo Numerics updated and rebuilt successfully!
```

### Alternative: NuGet Package

For the most convenient setup (no building required), use the NuGet package:
```bash
# Install the NuGet package to a local directory
nuget install RMC.Numerics -OutputDirectory ./packages
```

Then reference the DLL from `./packages/RMC.Numerics.{version}/lib/net8.0/Numerics.dll`

## Prerequisites
Before running this notebook, ensure you have:
1. **.NET Runtime** (6.0+ or .NET Framework 4.8.1 on Windows)
2. **Python 3.8+**
3. **Required packages** (install via pip):
   ```bash
   pip install pythonnet numpy matplotlib pandas
   ```
**Note:** Above are the most common Python packages used in this library, but the list is not comprehensive. 

## Step 1: Load the .NET Runtime
**CRITICAL:** You must load the .NET runtime BEFORE importing `clr`. This is the most common setup mistake.

In [1]:
import pythonnet

# Load the CoreCLR runtime (for .NET 6+)
# Use "netfx" instead if you're on Windows with .NET Framework
# If using .NET 4.8 skip this step!
pythonnet.load("coreclr")

print("✓ .NET runtime loaded successfully")

✓ .NET runtime loaded successfully


## Step 2: Import clr and Load Numerics DLL
Now we can import the CLR bridge and load the Numerics assmbly.

In [2]:
import clr 
from pathlib import Path 

# Path to your Numerics.dll
# MODIFY THIS PATH to make your installation
dll_path = Path(r"C:\GIT\Numerics\Numerics\bin\Debug\net8.0\Numerics.dll")

# Alternative: if Numerics.dll is in the same folder as this notebook:
# dll_path = Path("Numerics.dll")

if not dll_path.exists():
    raise FileNotFoundError(f"Cannot find Numerics.dll at {dll_path}")

clr.AddReference(str(dll_path))

print(f"✓ Loaded Numerics from: {dll_path}")

✓ Loaded Numerics from: C:\GIT\Numerics\Numerics\bin\Debug\net8.0\Numerics.dll


## Step 3: Import Numerics Namespaces
Now we can import classes from Numerics just like any other Python modeule. Be sure to check how the namespaces are organized within Numerics. Below we import the name spaces we need for a basic example of using the Normal Distibution.

In [4]:
# Import the most commonly used namespaces
from Numerics.Distributions import Normal, Uniform, GeneralizedExtremeValue
from Numerics.Sampling.MCMC import RWMH, MCMCResults
from Numerics.Mathematics.Optimization import NelderMead
from System import Array, Double

print("✓ Numerics namespaces imported successfully")

✓ Numerics namespaces imported successfully


## Basic Numerics Example: Normal Distribtion
Let's create a Normal Distribution and compute some basic statistics.

In [5]:
# Create a Normal Distribution wiht mean=100, sd=15
dist = Normal(100,15)

# Get statistical properties
print(f"Mean: {dist.Mean}")
print(f"Standard Deviation: {dist.StandardDeviation}")
print(f"Variance: {dist.Variance}")
print(f"Skewness: {dist.Skewness}")
print(f"Kurtosis: {dist.Kurtosis}")

Mean: 100.0
Standard Deviation: 15.0
Variance: 225.0
Skewness: 0.0
Kurtosis: 3.0


## Working with .NET Arrays
Many Numerics methods require .NET arrays instead of Python lists. Here's how to convert between them. There is also a helper function in src > helper_function.py > convert_to_dotnet_array(pythonList)

In [6]:
# Python list to .NET Array[Double]
python_data = [10.5, 20.3, 15.7, 18.9, 22.1]
dotnet_array = Array[Double](python_data)

print(f"Python list: {python_data}")
print(f".NET array length: {dotnet_array.Length}")

# .NET array back to Python list
back_to_python = [dotnet_array[i] for i in range(dotnet_array.Length)]
# Or more concisely:
back_to_python = list(dotnet_array)

print(f"Converted back: {back_to_python}")

Python list: [10.5, 20.3, 15.7, 18.9, 22.1]
.NET array length: 5
Converted back: [10.5, 20.3, 15.7, 18.9, 22.1]


## Common Problems and Solutions

### 1. Runtime Not Loaded Error
Make sure you loaded the runtime BEFORE importing clr.
```python
# ❌ WRONG - will fail
import clr
import pythonnet
pythonnet.load("coreclr")

# ✅ CORRECT - load runtime FIRST
import pythonnet
pythonnet.load("coreclr")
import clr
```
### 2. DLL not found
Verify the path in `dll_path` is correct.

### 3. Loading DLL more than once
When running multiple files at once, if they are all trying to load Numerics with the command 
```python
clr.AddReference(str(dll_path))
```
it may fail with an error that Numerics has already been loaded. If this happens, add yhe statement to help check the assembilies to see if Numerics has already been loaded. The code below ensures that the dll will only be loaded if 'Numerics' does not exist in the current assembly list.
```python
# Needed to access the assemblies
from System import AppDomain

assemblies = AppDomain.CurrentDomain.GetAssemblies()
assembly_names = [assembly.GetName().Name for assembly in assemblies]
if 'Numerics' not in assembly_names:
    clr.AddReference(str(dll_path))
```
Better safe than sorry!

### 4. Array Type Mismatch
Check that you're using .NET arrays where required.
```python
# ❌ WRONG - Python list
data = [1, 2, 3]
dist.LogLikelihood(data)  # Error!

# ✅ CORRECT - .NET array
data = Array[Double]([1, 2, 3])
dist.LogLikelihood(data)  # Works!
```

### 5. Platform Issues
Use `pythonnet.load("netfx")` on Windows with .NET Framework.

### 6. Accessing .NET Properties
```python
# ❌ WRONG - trying to call a property
mean_value = dist.Mean()  # Error!

# ✅ CORRECT - properties don't use parentheses
mean_value = dist.Mean
```

For detailed setup instructions, see `SETUP.md` in the repository.

## Next Steps
Now that you've learned the basics, explore the other notebooks:

- **01_distributions.ipynb** - Tour of 42+ probability distributions
- **02_distribution_fitting.ipynb** - Parameter estimation from data
- **03_mcmc_basics.ipynb** - Introduction to Bayesian inference
- **04_mcmc_bayesian_inference.ipynb** - Practical MCMC workflows
- **And more!**